In [1]:
import time

import torch
import torch.nn as nn

from torch_geometric.nn import GCNConv, SAGEConv
from torch_geometric.nn.pool import global_mean_pool
from torch.utils.data import ConcatDataset, random_split, DataLoader

from custom_logger import logger
from helpers_v2 import CVFConfigForGCNWSuccWEIDataset

In [2]:
device = "cuda"  # force cuda or exit

In [3]:
subset_size = 200_000

In [4]:
class SimpleGCN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.gcn1 = SAGEConv(input_size, hidden_size, bias=False)
        self.gcn2 = SAGEConv(hidden_size, hidden_size, bias=False)
        self.gcn3 = SAGEConv(hidden_size, hidden_size, bias=False)
        self.out = torch.nn.Linear(hidden_size, output_size)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x, edge_index):
        h = self.gcn1(x, edge_index)
        h = self.gcn2(h, edge_index)
        h = self.gcn3(h, edge_index)
        h = self.out(h)
        # print("h", h.shape)
        # h = torch.relu(h)
        h = global_mean_pool(h, torch.zeros(h.size(1)).to(device).long())
        h = self.sigmoid(h)
        return h

    def fit(self, epochs, dataloader):
        criterion = torch.nn.BCELoss()
        optimizer = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=0.00001)
        for epoch in range(1, epochs + 1):
            start_time = time.time()
            self.train()
            total_loss = 0
            count = 0
            for batch in dataloader:
                x = batch[0]
                y = batch[1]
                y = y.unsqueeze(-1)
                X = x[0].squeeze(0)
                out = self(X, x[1][0])
                optimizer.zero_grad()
                # print("out", out, "y", y)
                loss = criterion(out, y)
                total_loss += loss
                count += 1
                loss.backward()
                optimizer.step()

            logger.info(
                "Training set | Epoch: %s/%s | %s: %s | Time taken: %ss",
                epoch,
                epochs,
                criterion.__class__.__name__,
                round((total_loss / count).item(), 4),
                round(time.time() - start_time, 4),
            )

In [5]:
program = "graph_coloring"

graph_names = ["graph_powerlaw_cluster_graph_n8"]

H = 32

epochs = 500
batch_size = 64

In [6]:
def get_dataset_coll(program, *graph_names):
    dataset_coll = []

    for graph_name in graph_names:
        dataset_coll.append(
            CVFConfigForGCNWSuccWEIDataset(
                device,
                f"{graph_name}_config_rank_dataset.csv",
                f"{graph_name}_edge_index.json",
                program=program,
            )
        )

    return dataset_coll

In [7]:
dataset_coll = get_dataset_coll(program, *graph_names)
D = dataset_coll[0].D
train_sizes = [int(0.80 * len(ds)) for ds in dataset_coll]
test_sizes = [len(ds) - trs for ds, trs in zip(dataset_coll, train_sizes)]

train_test_datasets = [
    random_split(ds, [tr_s, ts])
    for ds, tr_s, ts in zip(dataset_coll, train_sizes, test_sizes)
]

train_datasets = [ds[0] for ds in train_test_datasets]
test_datasets = [ds[1] for ds in train_test_datasets]

datasets = ConcatDataset(train_datasets)



test_concat_datasets = ConcatDataset(test_datasets)

logger.info(
    f"Train dataset size: {len(datasets):,}, Subset size: {subset_size:,} | Test dataset size: {len(test_concat_datasets):,}"
)
logger.info("\n")

model = SimpleGCN(D, H, 1).to(device)
logger.info("Model %s", model)
logger.info(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Train dataset size: 114,688, Subset size: 200,000 | Test dataset size: 28,672


Model SimpleGCN(
  (gcn1): SAGEConv(1, 32, aggr=mean)
  (gcn2): SAGEConv(32, 32, aggr=mean)
  (gcn3): SAGEConv(32, 32, aggr=mean)
  (out): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Total parameters: 4,193


In [8]:
logger.info("\n")
start_time = time.time()
dataloader = DataLoader(datasets, batch_size=64)
model.fit(epochs=epochs, dataloader=dataloader)
logger.info("\n")
logger.info(
    "End Training | Total training time taken %ss",
    round(time.time() - start_time, 4),
)



Training set | Epoch: 1/500 | BCELoss: 0.3156 | Time taken: 8.5334s
Training set | Epoch: 2/500 | BCELoss: 0.2894 | Time taken: 8.3975s
Training set | Epoch: 3/500 | BCELoss: 0.2866 | Time taken: 8.4641s
Training set | Epoch: 4/500 | BCELoss: 0.2855 | Time taken: 8.4015s
Training set | Epoch: 5/500 | BCELoss: 0.2854 | Time taken: 8.414s
Training set | Epoch: 6/500 | BCELoss: 0.2851 | Time taken: 8.4142s
Training set | Epoch: 7/500 | BCELoss: 0.2871 | Time taken: 8.4156s
Training set | Epoch: 8/500 | BCELoss: 0.2845 | Time taken: 8.4162s
Training set | Epoch: 9/500 | BCELoss: 0.2864 | Time taken: 8.4749s
Training set | Epoch: 10/500 | BCELoss: 0.2846 | Time taken: 8.4057s
Training set | Epoch: 11/500 | BCELoss: 0.284 | Time taken: 8.4108s
Training set | Epoch: 12/500 | BCELoss: 0.2848 | Time taken: 8.4042s
Training set | Epoch: 13/500 | BCELoss: 0.2842 | Time taken: 8.4037s
Training set | Epoch: 14/500 | BCELoss: 0.285 | Time taken: 8.4665s
Training set | Epoch: 15/500 | BCELoss: 0.28

KeyboardInterrupt: 